In [46]:
!curl -O https://datasets.imdbws.com/title.basics.tsv.gz
!curl -O https://datasets.imdbws.com/title.ratings.tsv.gz
!curl -O https://datasets.imdbws.com/title.principals.tsv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  213M  100  213M    0     0  52.1M      0  0:00:04  0:00:04 --:--:-- 52.1M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8310k  100 8310k    0     0      0      0 --:--:-- --:--:-- --:--:--     0     0  25.8M      0 --:--:-- --:--:-- --:--:-- 25.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  735M  100  735M    0     0  55.4M      0  0:00:13  0:00:13 --:--:-- 54.5M


## Load the three IMDb datasets used in this analysis:
### title.basics (film metadata),
### title.ratings (ratings and vote counts),
### and title.principals (main cast and character names).

In [ ]:
import pandas as pd

basics = pd.read_csv("title.basics.tsv.gz", sep="\t", low_memory=False)
ratings = pd.read_csv("title.ratings.tsv.gz", sep="\t")
principals = pd.read_csv("title.principals.tsv.gz", sep="\t", low_memory=False)

In [48]:
basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [49]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2213
1,tt0000002,5.4,319
2,tt0000003,6.4,2347
3,tt0000004,5.1,192
4,tt0000005,6.2,3064


In [50]:
principals.head()

,tconst,ordering,nconst,category,job,characters
0,tt0000001,1,nm1588970,self,\N,"[""Self""]"
1,tt0000001,2,nm0005690,director,\N,\N
2,tt0000001,3,nm0005690,producer,producer,\N
3,tt0000001,4,nm0374658,cinematographer,director of photography,\N
4,tt0000002,1,nm0721526,director,\N,\N


In [51]:
basics["genres"].value_counts().head(20)

genres
Drama                1392679
Comedy                800635
Talk-Show             779352
News                  740207
Documentary           602796
Drama,Romance         559858
\N                    538949
Reality-TV            393611
Adult                 348179
News,Talk-Show        292406
Short                 243732
Drama,Short           238112
Family                220457
Game-Show             214494
Comedy,Short          175093
Documentary,Short     174866
Sport                 163358
Music                 127586
Romance               125483
Comedy,Talk-Show      121048
Name: count, dtype: int64

### Filter productions tagged with both Comedy and Romance on IMDb

In [52]:
# na=False treats null values as False instead of raising an error

romcoms = basics[
    basics["genres"].str.contains("Comedy", na=False) &
    basics["genres"].str.contains("Romance", na=False)
]

print(len(romcoms))
romcoms.head(10)

187839


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
245,tt0000248,short,A Kiss in the Tunnel,The Kiss in the Tunnel,0,1899,\N,1,"Comedy,Romance,Short"
448,tt0000452,short,The Messenger Boy's Mistake,The Messenger Boy's Mistake,0,1903,\N,2,"Comedy,Romance,Short"
760,tt0000770,short,The Taming of the Shrew,The Taming of the Shrew,0,1908,\N,17,"Comedy,Romance,Short"
923,tt0000933,short,Lady Helen's Escapade,Lady Helen's Escapade,0,1909,\N,8,"Comedy,Drama,Romance"
990,tt0001002,short,The Politician's Love Story,The Politician's Love Story,0,1909,\N,6,"Comedy,Romance,Short"
1013,tt0001025,short,A Rural Elopement,A Rural Elopement,0,1909,\N,9,"Comedy,Romance,Short"
1015,tt0001027,short,The Sacrifice,The Sacrifice,0,1909,\N,7,"Comedy,Romance,Short"
1298,tt0001310,short,May and December,May and December,0,1910,\N,7,"Comedy,Romance,Short"
1392,tt0001404,short,A Summer Flirtation,A Summer Flirtation,0,1910,\N,\N,"Comedy,Romance,Short"


In [53]:
romcoms["titleType"].value_counts()

titleType
tvEpisode       152434
movie            17771
short             9925
tvSeries          3278
tvMovie           2080
video             1355
tvMiniSeries       773
videoGame          144
tvShort             62
tvSpecial           17
Name: count, dtype: int64

### Filter just movies and tv movies.

In [72]:
romcoms = basics[
    basics["genres"].str.contains("Comedy", na=False) &
    basics["genres"].str.contains("Romance", na=False) &
    ((basics["titleType"] == "movie") | (basics["titleType"] == "tvMovie"))
]

print(len(romcoms))
romcoms.head(10)

KeyboardInterrupt: 

### Remove films with missing release years and keep only those released from 1980 onward

In [ ]:
romcoms = romcoms[romcoms["startYear"] != r"\N"]
romcoms = romcoms[romcoms["startYear"].astype(int) >= 1980]

print(len(romcoms))

13242


### Merge film metadata with IMDb ratings and keep only films with at least 10,000 user ratings

In [ ]:
romcoms_rated = pd.merge(romcoms, ratings, on="tconst")
romcoms_rated = romcoms_rated[romcoms_rated["numVotes"] > 10000]
romcoms_rated

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001,\N,118,"Comedy,Fantasy,Romance",6.4,94113
7,tt0079579,movie,Moscow Does Not Believe in Tears,Moskva slezam ne verit,0,1980,\N,150,"Comedy,Drama,Romance",8.0,15838
14,tt0080439,movie,The Taming of the Scoundrel,Il bisbetico domato,0,1980,\N,107,"Comedy,Romance",7.6,11494
40,tt0081480,movie,Seems Like Old Times,Seems Like Old Times,0,1980,\N,102,"Comedy,Romance",6.7,11883
55,tt0082031,movie,Arthur,Arthur,0,1981,\N,97,"Comedy,Romance",6.9,33723
...,...,...,...,...,...,...,...,...,...,...,...
11621,tt9731598,movie,Bros,Bros,0,2022,\N,115,"Comedy,Romance",6.3,39648
11638,tt9784456,movie,The Kissing Booth 2,The Kissing Booth 2,0,2020,\N,134,"Comedy,Romance",5.7,44266
11640,tt9806322,movie,Isi & Ossi,Isi & Ossi,0,2020,\N,113,"Comedy,Romance",6.4,11617
11656,tt9860728,movie,Falling Inn Love,Falling Inn Love,0,2019,\N,98,"Comedy,Romance",5.7,24227


In [ ]:
romcoms_rated["startYear"].value_counts()

startYear
2010    43
2008    42
2006    41
2014    38
2007    38
2011    37
2009    37
2004    35
2012    33
2005    31
2022    30
1999    29
2023    28
2015    27
2013    26
2018    26
2002    26
2000    24
2020    23
2016    23
2003    23
2001    23
2019    21
1995    20
2024    20
2017    19
2025    18
2021    17
1994    17
1998    16
1992    15
1996    15
1997    14
1987    13
1988    12
1993    11
1991    11
1990    10
1985     9
1986     9
1989     8
2026     7
1984     6
1980     5
1982     4
1983     4
1981     1
Name: count, dtype: int64

In [ ]:
principals["category"].value_counts()

category
actor                  23682694
actress                17890796
self                   15151560
writer                 11975508
director                8512088
producer                7429070
editor                  5262187
cinematographer         4040194
composer                3213290
production_designer     1190147
casting_director        1150373
archive_footage          642328
archive_sound             13544
Name: count, dtype: int64

### Keep only actors and actresses from the principals dataset. Merge the cast information with the filtered film dataset using tconst

In [ ]:
cast = principals[
    (principals["category"] == "actor") | (principals["category"] == "actress")
]

print(len(cast))

41573490


In [ ]:
romcoms_cast = pd.merge(romcoms_rated, cast, on="tconst")
print(len(romcoms_cast))

9951


#### Approximate the main protagonists by keeping only the first and second billed cast members.
#### This proved to be an imperfect approach: while ordering = 1 and 2 often corresponded to the main couple, many films featured supporting characters, ensemble casts, or billing
#### orders that did not reflect the romantic leads. These cases were reviewed and corrected manually in a later step.

In [ ]:
romcoms_cast = romcoms_cast[
    (romcoms_cast["ordering"] == 1) | (romcoms_cast["ordering"] == 2)
]

print(len(romcoms_cast))

1970


In [ ]:
romcoms_cast["characters"].value_counts().head(10)

characters
["Kate"]     10
["Nick"]      9
["Sam"]       8
["Jack"]      8
["Adam"]      6
["Peter"]     6
["Emma"]      6
["Tom"]       6
["Beth"]      6
["Ben"]       6
Name: count, dtype: int64

In [ ]:
romcoms_cast["characters"] = romcoms_cast["characters"].str.replace('["', '', regex=False).str.replace('"]', '', regex=False)

romcoms_cast["characters"]

0                         Kate McKay
1                            Leopold
10      Katerina 'Katya' Tikhomirova
11                   Georgiy 'Gosha'
20                      Elia Codogno
                    ...             
9922                         Isi (4)
9931                        Gabriela
9932                            Jake
9941                          Sloane
9942                         Jackson
Name: characters, Length: 1970, dtype: object

In [ ]:
romcoms_cast.to_csv("romcoms_cast.csv", index=False)

### First part of the analysis

In [ ]:
import pandas as pd

romcoms_cast = pd.read_csv("romcoms_cast.csv")

In [ ]:
len(romcoms_cast)
romcoms_cast["startYear"].max()

2026

In [ ]:
from openai import OpenAI
import json
import time

### The IMDb datasets do not include a character's profession.
#### To infer this information, I used the OpenAI API with a custom prompt that provided the movie title and the character's name. The model was instructed to infer a profession, occupation, social role, or primary activity rather than returning null whenever possible.

In [ ]:
client = OpenAI(
    api_key=""
)

In [75]:
def infer_profession_openai(title, character):

    prompt = f"""
Movie: {title}
Character: {character}

Identify this character's profession, occupation, social role, or primary activity.

Reasonable inference is allowed.

Examples:
doctor
writer
student
widow
single mother
princess
businessman
farmer

Return ONLY valid JSON in this format:
{{"profession": "short answer or null"}}

Return null only if absolutely nothing can be inferred.
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    text = response.output_text.strip()

    try:
        data = json.loads(text)
        return data["profession"]
    except:
        print("JSON error:", title, character)
        print(text)
        return None

In [76]:
new_professions = []

for index, row in romcoms_cast.iterrows():

    print(row["primaryTitle"], "-", row["characters"])

    profession = infer_profession_openai(
        row["primaryTitle"],
        row["characters"]
    )

    new_professions.append(profession)

    time.sleep(0.3)

Kate & Leopold - Kate McKay


KeyboardInterrupt: 

#### Save the inferred professions in a new column and export the complete dataset for manual review as "romcoms_professions_complete.csv".

In [ ]:
romcoms_cast["profession_final"] = new_professions

romcoms_cast.to_csv(
    "romcoms_professions_complete.csv",
    index=False
)

In [ ]:
romcoms_cast["profession_final"].isna().sum()

np.int64(69)

In [ ]:
romcoms_cast["profession_final"].value_counts().head(20)

profession_final
student                  343
null                     277
businessman              170
writer                    60
teacher                   49
businesswoman             32
artist                    28
single mother             25
journalist                23
lawyer                    21
actor                     19
wedding planner           19
advertising executive     16
chef                      16
widow                     15
doctor                    14
architect                 14
actress                   14
mother                    13
waitress                  13
Name: count, dtype: int64